## Step 1: Identify the Task

The objective of this project is to develop a Knowledge-Based Agent that can autonomously navigate a network of Kenyan roads from a starting location to a specified destination.

The agent will use knowledge about towns, road connections, distances, and road conditions to make informed navigation decisions.

The system builds upon Practical Exercise 3, which implemented Uniform Cost Search (UCS) to identify the shortest-distance route between Nairobi and Kisumu.

In this exercise, the existing search algorithm will be enhanced with a Knowledge Base and an Inference Engine. The Knowledge Base will store information about the road network and its conditions, while the Inference Engine will apply logical rules to determine suitable routes.

The agent should be capable of:

1. Identifying its current location and destination.
2. Retrieving information about connected towns and road distances.
3. Determining whether roads are open, closed, or unsafe.
4. Selecting the most suitable route based on available knowledge.
5. Avoiding roads that are closed or unsafe.
6. Recalculating its route when road conditions change.
7. Responding to user queries about locations, roads, and routes.

**Expected outcome:** A knowledge-based self-driving trailer agent capable of reasoning about road conditions and selecting an appropriate route to reach its destination.

## Step 2: Assemble the Relevant Knowledge

The knowledge required by the self-driving trailer agent is gathered from the existing implementation in Practical Exercise 3 and additional road-condition information.

The existing implementation provides a directed road graph containing towns as nodes and roads as weighted edges, where the weights represent distances in kilometres.

Additional knowledge is introduced to describe road conditions, including whether a road is open, closed, or unsafe.

The knowledge sources include:

1. **Practical Exercise 3:** Existing search tree, directed graph and distance measurements.
2. **Road network information:** Connections between towns and their corresponding distances.
3. **Road-condition information:** Information about road closures and safety conditions.
4. **User input:** Starting location, destination and updated road observations.

For this implementation, road-condition information will initially be simulated. The knowledge base can later be extended using external APIs or web-scraped information where permitted.


### Visual Representation of the Road Network

The following directed graph represents the road network used by the self-driving trailer agent. Each road connection is bidirectional, meaning the trailer can travel in either direction. The edge labels represent the distance between towns in kilometres.

```mermaid
graph TD
    Nairobi <-->|30 km| Limuru
    Nairobi <-->|45 km| Thika
    Nairobi <-->|140 km| Narok

    Limuru <-->|60 km| Naivasha

    Thika <-->|40 km| Muranga
    Muranga <-->|55 km| Nyeri
    Nyeri <-->|70 km| Nyahururu

    Naivasha <-->|30 km| Gilgil
    Gilgil <-->|35 km| Nakuru
    Nyahururu <-->|85 km| Nakuru

    Nakuru <-->|75 km| Londiani
    Londiani <-->|35 km| Kericho
    Londiani <-->|85 km| Eldoret

    Narok <-->|95 km| Bomet
    Bomet <-->|50 km| Kericho

    Kericho <-->|90 km| Kisumu
```


In [ ]:
# Each road is represented as (Town A, Town B, Distance in km)

ROADS = [
    ("Nairobi", "Limuru", 30),
    ("Nairobi", "Thika", 45),
    ("Nairobi", "Narok", 140),
    ("Limuru", "Naivasha", 60),
    ("Thika", "Murang'a", 40),
    ("Murang'a", "Nyeri", 55),
    ("Nyeri", "Nyahururu", 70),
    ("Naivasha", "Gilgil", 30),
    ("Gilgil", "Nakuru", 35),
    ("Nyahururu", "Nakuru", 85),
    ("Nakuru", "Londiani", 75),
    ("Londiani", "Kericho", 35),
    ("Londiani", "Eldoret", 85),
    ("Narok", "Bomet", 95),
    ("Bomet", "Kericho", 50),
    ("Kericho", "Kisumu", 90)
]

# 2. Geographical information about the towns

REGIONS = {
    "Nairobi": "Nairobi",
    "Limuru": "Central Highlands",
    "Thika": "Central Highlands",
    "Murang'a": "Central Highlands",
    "Nyeri": "Central Highlands",
    "Nyahururu": "Central Highlands",
    "Naivasha": "Rift Valley",
    "Gilgil": "Rift Valley",
    "Nakuru": "Rift Valley",
    "Londiani": "Rift Valley",
    "Kericho": "Rift Valley",
    "Eldoret": "Rift Valley",
    "Narok": "Rift Valley",
    "Bomet": "Rift Valley",
    "Kisumu": "Lake Victoria Basin"
}

# 3. Define the starting point and destination

INITIAL_STATE = "Nairobi"
GOAL_STATE = "Kisumu"

# 4. Build the road network graph

def build_graph(roads):
    graph = {}

    for town_a, town_b, distance in roads:
        graph.setdefault(town_a, []).append((town_b, distance))
        graph.setdefault(town_b, []).append((town_a, distance))

    return graph


graph = build_graph(ROADS)

# 5. Introduce initial road-condition knowledge
# These are simulated conditions for testing purposes.

road_conditions = {}

for town_a, town_b, distance in ROADS:
    road_conditions[frozenset([town_a, town_b])] = "Open"

# 6. Display the collected knowledge

print("=" * 65)
print("SELF-DRIVING TRAILER - KNOWLEDGE COLLECTION")
print("=" * 65)

print("\nROAD NETWORK:")
for town, connections in graph.items():
    print(f"{town:<12} -> {connections}")

print("\nGEOGRAPHICAL INFORMATION:")
for town, region in REGIONS.items():
    print(f"{town:<12} -> {region}")

print("\nROAD CONDITIONS:")
for road, condition in road_conditions.items():
    towns = sorted(road)
    print(f"{towns[0]} <-> {towns[1]} : {condition}")

print("\nKNOWLEDGE SUMMARY")
print("-" * 65)
print("Total towns:", len(REGIONS))
print("Total road connections:", len(ROADS))
print("Total directed edges:", sum(len(v) for v in graph.values()))
print("Starting location:", INITIAL_STATE)
print("Destination:", GOAL_STATE)
print("Road condition source: Simulated initial information")


SELF-DRIVING TRAILER - KNOWLEDGE COLLECTION

ROAD NETWORK:
Nairobi      -> [('Limuru', 30), ('Thika', 45), ('Narok', 140)]
Limuru       -> [('Nairobi', 30), ('Naivasha', 60)]
Thika        -> [('Nairobi', 45), ("Murang'a", 40)]
Narok        -> [('Nairobi', 140), ('Bomet', 95)]
Naivasha     -> [('Limuru', 60), ('Gilgil', 30)]
Murang'a     -> [('Thika', 40), ('Nyeri', 55)]
Nyeri        -> [("Murang'a", 55), ('Nyahururu', 70)]
Nyahururu    -> [('Nyeri', 70), ('Nakuru', 85)]
Gilgil       -> [('Naivasha', 30), ('Nakuru', 35)]
Nakuru       -> [('Gilgil', 35), ('Nyahururu', 85), ('Londiani', 75)]
Londiani     -> [('Nakuru', 75), ('Kericho', 35), ('Eldoret', 85)]
Kericho      -> [('Londiani', 35), ('Bomet', 50), ('Kisumu', 90)]
Eldoret      -> [('Londiani', 85)]
Bomet        -> [('Narok', 95), ('Kericho', 50)]
Kisumu       -> [('Kericho', 90)]

GEOGRAPHICAL INFORMATION:
Nairobi      -> Nairobi
Limuru       -> Central Highlands
Thika        -> Central Highlands
Murang'a     -> Central Highlands



## Step 3: Define the Vocabulary

In this step, we define the vocabulary that will be used by the knowledge-based self-driving trailer agent.

The vocabulary identifies the important entities, properties and relationships within the road network.

### 1. Entities

- **Town:** A location within the road network, such as Nairobi, Nakuru or Kisumu.
- **Road:** A connection between two towns.
- **Distance:** The distance between two connected towns, measured in kilometres.
- **Region:** The geographical region where a town is located.
- **Road Condition:** The current state of a road, which can be Open, Closed or Unsafe.
- **Route:** A sequence of connected towns that the trailer can follow to reach its destination.

### 2. Predicates (Relationships)

The following predicates describe relationships and facts about the road network:

- `Connected(A, B)`: Town A is connected to town B.
- `Distance(A, B, D)`: The distance between towns A and B is D kilometres.
- `LocatedIn(A, R)`: Town A is located in region R.
- `RoadStatus(A, B, S)`: The road between towns A and B has status S.
- `At(A)`: The trailer is currently at town A.
- `Destination(A)`: Town A is the trailer's destination.
- `Safe(A, B)`: The road connecting towns A and B is safe to travel on.

### 3. Road Conditions

The agent will recognise three possible road conditions:

- **Open:** The road is available for travel.
- **Closed:** The road is unavailable and must be avoided.
- **Unsafe:** The road has a safety concern and should be avoided.

These terms will be used when encoding facts and rules into the knowledge base in the following steps.


In [2]:
# Define the agent's initial state
AGENT_STATE = {
    "current_location": INITIAL_STATE,
    "destination": GOAL_STATE
}



## Step 4: Encode General Knowledge

In this step, we encode general knowledge about road navigation into the knowledge base.

The agent uses predefined rules to determine which roads it can travel on and which roads it should avoid.

These rules provide the general knowledge that the agent will use when reasoning about the road network.


In [3]:
# Store the general knowledge rules
GENERAL_RULES = {
    "RoadConnectivity": "Two towns are connected if a road exists between them.",
    "SafeRoad": "A road is safe if its condition is Open.",
    "ClosedRoad": "A Closed road cannot be used.",
    "UnsafeRoad": "An Unsafe road should be avoided.",
    "Travel": "The agent can travel if towns are connected and the road is Open.",
    "Distance": "The total route distance is the sum of its road distances."
}


# Rule 1: Check whether two towns are connected
def is_connected(town_a, town_b):
    return any(
        neighbour == town_b
        for neighbour, distance in graph.get(town_a, [])
    )


# Rule 2: Retrieve the condition of a road
def get_road_status(town_a, town_b):
    road = frozenset([town_a, town_b])
    return road_conditions.get(road, "Unknown")


# Rule 3: Check whether a road is safe
def is_safe(town_a, town_b):
    return get_road_status(town_a, town_b) == "Open"


# Rule 4: Determine whether the agent can travel between two towns
def can_travel(town_a, town_b):
    return is_connected(town_a, town_b) and is_safe(town_a, town_b)


# Rule 5: Find all safe neighbouring towns
def get_safe_neighbors(town):
    safe_neighbors = []

    for neighbour, distance in graph.get(town, []):
        if can_travel(town, neighbour):
            safe_neighbors.append((neighbour, distance))

    return safe_neighbors


# Rule 6: Calculate the total distance of a route
def calculate_route_distance(route):
    total_distance = 0

    for i in range(len(route) - 1):
        town_a = route[i]
        town_b = route[i + 1]

        if not is_connected(town_a, town_b):
            return None

        total_distance += next(
            distance
            for neighbour, distance in graph[town_a]
            if neighbour == town_b
        )

    return total_distance


# Display the encoded general knowledge
print("========== GENERAL KNOWLEDGE RULES ==========")

for rule, description in GENERAL_RULES.items():
    print(f"{rule}: {description}")


# Test the general knowledge rules
print("\n========== TESTING THE RULES ==========")

print("Are Nairobi and Limuru connected?",
      is_connected("Nairobi", "Limuru"))

print("Is the road between Nairobi and Limuru safe?",
      is_safe("Nairobi", "Limuru"))

print("Can the trailer travel from Nairobi to Limuru?",
      can_travel("Nairobi", "Limuru"))

print("Safe neighbours of Nairobi:",
      get_safe_neighbors("Nairobi"))

sample_route = ["Nairobi", "Limuru", "Naivasha"]

print("Distance of sample route:",
      calculate_route_distance(sample_route), "km")



# Test different road conditions
print("\n========== ROAD CONDITION ANALYSIS ==========")

test_road = frozenset(["Nairobi", "Limuru"])

# Simulate a closed road
road_conditions[test_road] = "Unsafe"

town_a = "Nairobi"
town_b = "Limuru"

status = get_road_status(town_a, town_b)

print(f"\nThe road from {town_a} to {town_b} is {status.lower()}.")

if status == "Open":
    print("The road is open for use.")
    print("The trailer can travel.")
elif status == "Closed":
    print("The road is unavailable and unsafe for travel.")
    print("The trailer cannot travel.")
elif status == "Unsafe":
    print("The road is open but unsafe.")
    print("The trailer cannot travel.")
else:
    print("The road condition is unknown.")
    print("The trailer cannot travel.")

# Restore the original road condition
road_conditions[test_road] = "Open"


========== GENERAL KNOWLEDGE RULES ==========
RoadConnectivity: Two towns are connected if a road exists between them.
SafeRoad: A road is safe if its condition is Open.
ClosedRoad: A Closed road cannot be used.
UnsafeRoad: An Unsafe road should be avoided.
Travel: The agent can travel if towns are connected and the road is Open.
Distance: The total route distance is the sum of its road distances.

========== TESTING THE RULES ==========
Are Nairobi and Limuru connected? True
Is the road between Nairobi and Limuru safe? True
Can the trailer travel from Nairobi to Limuru? True
Safe neighbours of Nairobi: [('Limuru', 30), ('Thika', 45), ('Narok', 140)]
Distance of sample route: 90 km

========== ROAD CONDITION ANALYSIS ==========

The road from Nairobi to Limuru is unsafe.
The road is open but unsafe.
The trailer cannot travel.



## Step 5: Add New Facts to the Knowledge Base

In this step, we populate the knowledge base with specific facts about the road network and the self-driving trailer's environment.

The facts are obtained from the road network, town regions, distances and initial road conditions defined in Step 2.

### Facts to be stored

1. **Town Facts:** The towns that exist in the road network.
2. **Connectivity Facts:** The connections between towns.
3. **Distance Facts:** The distances between connected towns in kilometres.
4. **Region Facts:** The geographical regions where towns are located.
5. **Road Condition Facts:** The current condition of each road.
6. **Agent State Facts:** The trailer's current location and destination.

Since the roads are bidirectional, connections and distances will be stored in both directions.

These facts will form the knowledge base that the agent will use when answering queries and making navigation decisions.


In [4]:
# 1. Create the knowledge base
KNOWLEDGE_BASE = {
    "Town": set(),
    "Connected": set(),
    "Distance": set(),
    "LocatedIn": set(),
    "RoadStatus": set(),
    "At": set(),
    "Destination": set()
}


# 2. Function for adding facts to the knowledge base
def add_fact(predicate, fact):
    KNOWLEDGE_BASE[predicate].add(fact)


# 3. Add town facts
for town in graph:
    add_fact("Town", town)


# 4. Add connectivity, distance and road condition facts
for town_a, town_b, distance in ROADS:

    # Store connections in both directions
    add_fact("Connected", (town_a, town_b))
    add_fact("Connected", (town_b, town_a))

    # Store distances in both directions
    add_fact("Distance", (town_a, town_b, distance))
    add_fact("Distance", (town_b, town_a, distance))

    # Store current road conditions in both directions
    status = road_conditions[frozenset([town_a, town_b])]

    add_fact("RoadStatus", (town_a, town_b, status))
    add_fact("RoadStatus", (town_b, town_a, status))


# 5. Add geographical region facts
for town, region in REGIONS.items():
    add_fact("LocatedIn", (town, region))


# 6. Add the trailer's current location and destination
add_fact("At", INITIAL_STATE)
add_fact("Destination", GOAL_STATE)


# 7. Display the knowledge base
print("========== KNOWLEDGE BASE ==========")

for predicate, facts in KNOWLEDGE_BASE.items():

    print(f"\n{predicate}:")

    for fact in sorted(facts):
        print(fact)


# 8. Display knowledge base statistics
print("\n========== KNOWLEDGE BASE SUMMARY ==========")

for predicate, facts in KNOWLEDGE_BASE.items():
    print(f"{predicate}: {len(facts)} facts")

print("\nTotal facts:",
      sum(len(facts) for facts in KNOWLEDGE_BASE.values()))


========== KNOWLEDGE BASE ==========

Town:
Bomet
Eldoret
Gilgil
Kericho
Kisumu
Limuru
Londiani
Murang'a
Nairobi
Naivasha
Nakuru
Narok
Nyahururu
Nyeri
Thika

Connected:
('Bomet', 'Kericho')
('Bomet', 'Narok')
('Eldoret', 'Londiani')
('Gilgil', 'Naivasha')
('Gilgil', 'Nakuru')
('Kericho', 'Bomet')
('Kericho', 'Kisumu')
('Kericho', 'Londiani')
('Kisumu', 'Kericho')
('Limuru', 'Nairobi')
('Limuru', 'Naivasha')
('Londiani', 'Eldoret')
('Londiani', 'Kericho')
('Londiani', 'Nakuru')
("Murang'a", 'Nyeri')
("Murang'a", 'Thika')
('Nairobi', 'Limuru')
('Nairobi', 'Narok')
('Nairobi', 'Thika')
('Naivasha', 'Gilgil')
('Naivasha', 'Limuru')
('Nakuru', 'Gilgil')
('Nakuru', 'Londiani')
('Nakuru', 'Nyahururu')
('Narok', 'Bomet')
('Narok', 'Nairobi')
('Nyahururu', 'Nakuru')
('Nyahururu', 'Nyeri')
('Nyeri', "Murang'a")
('Nyeri', 'Nyahururu')
('Thika', "Murang'a")
('Thika', 'Nairobi')

Distance:
('Bomet', 'Kericho', 50)
('Bomet', 'Narok', 95)
('Eldoret', 'Londiani', 85)
('Gilgil', 'Naivasha', 30)
('Gilgi


## Step 6: Encode the Specific Problem Instance (User Input)

In this step, we allow the user to interact with the knowledge-based self-driving trailer agent by providing information about a specific navigation problem.

The agent requests the following information:

1. **Current Location:** The town where the trailer is currently located.
2. **Destination:** The town the trailer needs to reach.
3. **Road Condition:** The user can report a road as Open, Closed or Unsafe.

The agent validates the information provided and updates its knowledge base accordingly.

After receiving the information, the agent responds by:

- Confirming its current location and destination.
- Acknowledging the reported road condition.
- Explaining whether the reported road can be used.
- Identifying the available roads from its current location.

This makes the agent interactive and allows it to respond to changing road conditions instead of relying entirely on predefined information.


In [5]:
import heapq
import re


# NORMALISE TOWN NAMES
def normalise_town_name(name):
    return name.lower().replace("'", "").strip()


town_lookup = {
    normalise_town_name(town): town
    for town in KNOWLEDGE_BASE["Town"]
}


# FIND THE SHORTEST ROUTE, IGNORING ROAD CONDITIONS
def find_route(start, goal):
    frontier = [(0, [start])]
    visited = {}

    while frontier:
        distance, path = heapq.heappop(frontier)
        current = path[-1]

        if current == goal:
            return path, distance

        if current in visited and visited[current] <= distance:
            continue

        visited[current] = distance

        for neighbour, road_distance in graph.get(current, []):
            new_distance = distance + road_distance

            if neighbour not in visited or new_distance < visited[neighbour]:
                heapq.heappush(
                    frontier,
                    (new_distance, path + [neighbour])
                )

    return None, None


# FIND THE SHORTEST SAFE ROUTE
def find_best_route(start, goal):
    frontier = [(0, [start])]
    visited = {}

    while frontier:
        distance, path = heapq.heappop(frontier)
        current = path[-1]

        if current == goal:
            return path, distance

        if current in visited and visited[current] <= distance:
            continue

        visited[current] = distance

        for neighbour, road_distance in graph.get(current, []):

            status = road_conditions.get(
                frozenset([current, neighbour]), "Unknown"
            )

            if status != "Open":
                continue

            new_distance = distance + road_distance

            if neighbour not in visited or new_distance < visited[neighbour]:
                heapq.heappush(
                    frontier,
                    (new_distance, path + [neighbour])
                )

    return None, None

while True:
    current_input = input("Enter current location: ").strip()
    current_key = normalise_town_name(current_input)

    if current_key in town_lookup:
        current_location = town_lookup[current_key]
        break

    print("Invalid location. Please enter a town in the road network.")


while True:
    destination_input = input("Enter destination: ").strip()
    destination_key = normalise_town_name(destination_input)

    if destination_key in town_lookup:
        destination = town_lookup[destination_key]

        if destination != current_location:
            break

        print("Destination cannot be the same as current location.")

    else:
        print("Invalid destination. Please enter a town in the road network.")


while True:
    road_input = input(
        "Which road or route is closed or unsafe? "
        "(e.g. Naivasha to Nakuru is closed): "
    ).strip()

    match = re.fullmatch(
        r"(.+?)\s+to\s+(.+?)\s+is\s+(closed|unsafe)",
        road_input,
        re.IGNORECASE
    )

    if not match:
        print("Please use the format: Town A to Town B is closed/unsafe.")
        continue

    town_a_key = normalise_town_name(match.group(1))
    town_b_key = normalise_town_name(match.group(2))
    road_status = match.group(3).capitalize()

    if town_a_key not in town_lookup or town_b_key not in town_lookup:
        print("Invalid town. Please try again.")
        continue

    town_a = town_lookup[town_a_key]
    town_b = town_lookup[town_b_key]

    if town_a == town_b:
        print("Please enter two different towns.")
        continue

    # Find the route between the reported towns
    reported_path, reported_distance = find_route(town_a, town_b)

    if reported_path is None:
        print("No route exists between those towns.")
        continue

    break

AGENT_STATE["current_location"] = current_location
AGENT_STATE["destination"] = destination

PROBLEM_INSTANCE = {
    "initial_location": current_location,
    "destination": destination,
    "reported_route": reported_path,
    "road_status": road_status
}

KNOWLEDGE_BASE["At"].clear()
KNOWLEDGE_BASE["Destination"].clear()

add_fact("At", current_location)
add_fact("Destination", destination)


# Reset road conditions before applying the new report
for a, b, distance in ROADS:
    road_conditions[frozenset([a, b])] = "Open"

KNOWLEDGE_BASE["RoadStatus"].clear()

for a, b, distance in ROADS:
    add_fact("RoadStatus", (a, b, "Open"))
    add_fact("RoadStatus", (b, a, "Open"))


# Apply the reported condition to every road section
# along the identified route
for i in range(len(reported_path) - 1):
    a = reported_path[i]
    b = reported_path[i + 1]

    road_conditions[frozenset([a, b])] = road_status

    KNOWLEDGE_BASE["RoadStatus"].discard((a, b, "Open"))
    KNOWLEDGE_BASE["RoadStatus"].discard((b, a, "Open"))

    add_fact("RoadStatus", (a, b, road_status))
    add_fact("RoadStatus", (b, a, road_status))

if len(reported_path) == 2:
    print(f"\nThe road from {town_a} to {town_b} is {road_status.lower()}.")
else:
    print(
        f"\nThe route from {town_a} to {town_b} is {road_status.lower()}."
    )

print("I will avoid using this route.")

best_path, best_distance = find_best_route(
    current_location, destination
)

if best_path:
    print("\nI recommend using this route:")
    print(" -> ".join(best_path))
    print(f"Total distance: {best_distance} km")

else:
    print("\nNo available safe route to the destination.")
    print("The trailer cannot proceed.")



The route from Naivasha to Nakuru is closed.
I will avoid using this route.

I recommend using this route:
Nairobi -> Narok -> Bomet -> Kericho -> Londiani -> Eldoret
Total distance: 405 km


**Step 7: Pose Queries and get Answers**

The agent can answer questions about the road network using facts stored in its knowledge base. These include town locations, road connections, distances, road conditions, the current location, and the destination. The agent uses these facts to respond to user queries.

In [7]:
import re
import heapq

def get_towns_from_question(question):
    found_towns = []

    for town in KNOWLEDGE_BASE["Town"]:
        pattern = r"\b" + re.escape(town) + r"\b"

        match = re.search(pattern, question, re.IGNORECASE)

        if match:
            found_towns.append((match.start(), town))

    found_towns.sort(key=lambda item: item[0])

    return [town for position, town in found_towns]


# FIND THE SHORTEST AVAILABLE ROUTE USING UCS
def find_shortest_safe_route(start, goal):
    frontier = [(0, [start])]
    visited = {}

    while frontier:
        distance, path = heapq.heappop(frontier)
        current = path[-1]

        if current == goal:
            return path, distance

        if current in visited and visited[current] <= distance:
            continue

        visited[current] = distance

        for neighbour, road_distance in graph.get(current, []):

            if not is_safe(current, neighbour):
                continue

            new_distance = distance + road_distance

            if neighbour not in visited or new_distance < visited[neighbour]:
                heapq.heappush(
                    frontier,
                    (new_distance, path + [neighbour])
                )

    return None, None

def answer_query(question):

    question_lower = question.lower().strip()

    # CURRENT LOCATION
    if "current location" in question_lower or "where am i" in question_lower:
        return f"You are currently at {AGENT_STATE['current_location']}."

    # DESTINATION
    if "destination" in question_lower or "where am i going" in question_lower:
        return f"Your destination is {AGENT_STATE['destination']}."

    # NUMBER OF TOWNS
    if "how many towns" in question_lower:
        return f"The road network contains {len(KNOWLEDGE_BASE['Town'])} towns."

    # CLOSED ROADS
    if "closed" in question_lower:
        closed_roads = set()

        for a, b, status in KNOWLEDGE_BASE["RoadStatus"]:
            if status == "Closed":
                closed_roads.add(tuple(sorted([a, b])))

        if closed_roads:
            roads = [
                f"{a} to {b}"
                for a, b in sorted(closed_roads)
            ]
            return "The following roads are closed: " + ", ".join(roads) + "."

        return "There are currently no closed roads."

    # UNSAFE ROADS
    if "unsafe" in question_lower:
        unsafe_roads = set()

        for a, b, status in KNOWLEDGE_BASE["RoadStatus"]:
            if status == "Unsafe":
                unsafe_roads.add(tuple(sorted([a, b])))

        if unsafe_roads:
            roads = [
                f"{a} to {b}"
                for a, b in sorted(unsafe_roads)
            ]
            return "The following roads are unsafe: " + ", ".join(roads) + "."

        return "There are currently no unsafe roads."

    # OPEN ROADS
    if "open roads" in question_lower:
        open_roads = set()

        for a, b, status in KNOWLEDGE_BASE["RoadStatus"]:
            if status == "Open":
                open_roads.add(tuple(sorted([a, b])))

        if open_roads:
            roads = [
                f"{a} to {b}"
                for a, b in sorted(open_roads)
            ]
            return "The following roads are open: " + ", ".join(roads) + "."

        return "There are currently no open roads."

    # IDENTIFY TOWNS
    towns = get_towns_from_question(question)

    if len(towns) < 2:
        if "distance" in question_lower or "route" in question_lower:
            return "Please mention two towns in your question."

        return "Sorry, I cannot answer that question."

    town_a, town_b = towns[0], towns[1]

    # DISTANCE AND ROUTE QUERIES
    if "distance" in question_lower or "shortest route" in question_lower:

        route, distance = find_shortest_safe_route(town_a, town_b)

        if route:
            route_text = " -> ".join(route)

            return (
                f"The shortest available route is {route_text} "
                f"({distance} km)."
            )

        return f"No available safe route exists between {town_a} and {town_b}."

    # ROAD CONDITION QUERIES
    if "status" in question_lower or "condition" in question_lower:

        # Find the shortest physical route, ignoring road conditions
        route, distance = find_route(town_a, town_b)

        if route is None:
            return f"No road route exists between {town_a} and {town_b}."

        # Check the condition of every road section on that route
        statuses = []

        for i in range(len(route) - 1):
            a = route[i]
            b = route[i + 1]

            status = road_conditions.get(
                frozenset([a, b]), "Unknown"
            )

            statuses.append(status)

        # All sections have the same condition
        if all(status == "Open" for status in statuses):
            return (
                f"The route between {town_a} and {town_b} is open."
            )

        if all(status == "Closed" for status in statuses):
            return (
                f"The route between {town_a} and {town_b} is closed."
            )

        if all(status == "Unsafe" for status in statuses):
            return (
                f"The route between {town_a} and {town_b} is unsafe."
            )

        # Mixed road conditions
        restrictions = []

        for i in range(len(route) - 1):
            if statuses[i] != "Open":
                restrictions.append(
                    f"{route[i]} to {route[i + 1]} is "
                    f"{statuses[i].lower()}"
                )

        return (
            f"The route between {town_a} and {town_b} "
            f"has restrictions: {'; '.join(restrictions)}."
        )

    if "direct road" in question_lower:

        if (town_a, town_b) in KNOWLEDGE_BASE["Connected"]:
            return f"{town_a} and {town_b} have a direct road connection."

        return f"There is no direct road between {town_a} and {town_b}."

    if "connected" in question_lower:

        route, distance = find_shortest_safe_route(town_a, town_b)

        if route:
            return f"Yes. They are connected through {' -> '.join(route)}."

        return f"There is no available safe route between {town_a} and {town_b}."

    return "Sorry, I cannot answer that question."

print("Ask the agent a question (type 'exit' to stop).")

while True:

    user_question = input("\nYou: ").strip()

    if user_question.lower() == "exit":
        print("Agent: Goodbye!")
        break

    print("Agent:", answer_query(user_question))


Ask the agent a question (type 'exit' to stop).


Agent: The route between Naivasha and Nakuru is closed.
Agent: Goodbye!


**Step 8: Query the knowledge base.**

The agent is evaluated using different road-network scenarios to determine how effectively it responds to changing environmental conditions. Each scenario introduces one or more road closures or unsafe roads. The agent updates its knowledge base, evaluates available routes using Uniform Cost Search, and selects the shortest safe route to its destination.

The scenarios demonstrate the agent's ability to adapt to road disruptions, avoid unsafe routes, select alternative routes and identify situations where no safe route is available.

In [8]:
import heapq


# FIND SHORTEST ROUTE WITHOUT CONSIDERING ROAD CONDITIONS
def find_physical_route(start, goal):

    frontier = [(0, [start])]
    best_distances = {start: 0}

    while frontier:
        distance, path = heapq.heappop(frontier)
        current = path[-1]

        if distance > best_distances.get(current, float("inf")):
            continue

        if current == goal:
            return path, distance

        for neighbour, road_distance in graph.get(current, []):

            new_distance = distance + road_distance

            if new_distance < best_distances.get(
                neighbour, float("inf")
            ):
                best_distances[neighbour] = new_distance

                heapq.heappush(
                    frontier,
                    (new_distance, path + [neighbour])
                )

    return None, None

def update_scenario_road(start, goal, status):

    route, distance = find_physical_route(start, goal)

    if route is None:
        return []

    affected_roads = []

    for i in range(len(route) - 1):

        a = route[i]
        b = route[i + 1]

        road_conditions[frozenset([a, b])] = status

        KNOWLEDGE_BASE["RoadStatus"].discard((a, b, "Open"))
        KNOWLEDGE_BASE["RoadStatus"].discard((b, a, "Open"))
        KNOWLEDGE_BASE["RoadStatus"].discard((a, b, "Closed"))
        KNOWLEDGE_BASE["RoadStatus"].discard((b, a, "Closed"))
        KNOWLEDGE_BASE["RoadStatus"].discard((a, b, "Unsafe"))
        KNOWLEDGE_BASE["RoadStatus"].discard((b, a, "Unsafe"))

        add_fact("RoadStatus", (a, b, status))
        add_fact("RoadStatus", (b, a, status))

        affected_roads.append(f"{a} to {b}")

    return affected_roads

def scenario_agent_decision(start, goal):

    frontier = [(0, [start])]
    best_distances = {start: 0}

    while frontier:

        distance, path = heapq.heappop(frontier)
        current = path[-1]

        if distance > best_distances.get(current, float("inf")):
            continue

        if current == goal:
            return path, distance

        for neighbour, road_distance in graph.get(current, []):

            status = road_conditions.get(
                frozenset([current, neighbour]), "Unknown"
            )

            if status != "Open":
                continue

            new_distance = distance + road_distance

            if new_distance < best_distances.get(
                neighbour, float("inf")
            ):
                best_distances[neighbour] = new_distance

                heapq.heappush(
                    frontier,
                    (new_distance, path + [neighbour])
                )

    return None, None


# SCENARIOS TO TEST
scenarios = [
    {
        "name": "Scenario 1: All roads are open",
        "reports": []
    },
    {
        "name": "Scenario 2: Naivasha to Nakuru is closed",
        "reports": [
            ("Naivasha", "Nakuru", "Closed")
        ]
    },
    {
        "name": "Scenario 3: Narok to Kericho is closed",
        "reports": [
            ("Narok", "Kericho", "Closed")
        ]
    },
    {
        "name": "Scenario 4: Both routes are closed",
        "reports": [
            ("Naivasha", "Nakuru", "Closed"),
            ("Narok", "Kericho", "Closed")
        ]
    },
    {
        "name": "Scenario 5: Naivasha to Nakuru is unsafe",
        "reports": [
            ("Naivasha", "Nakuru", "Unsafe")
        ]
    },
    {
        "name": "Scenario 6: All roads leaving Nairobi are closed",
        "reports": [
            ("Nairobi", "Limuru", "Closed"),
            ("Nairobi", "Thika", "Closed"),
            ("Nairobi", "Narok", "Closed")
        ]
    }
]


# SAVE ORIGINAL ROAD CONDITIONS
original_conditions = road_conditions.copy()
original_road_facts = KNOWLEDGE_BASE["RoadStatus"].copy()


# RUN ALL SCENARIOS
print("SELF-DRIVING TRAILER: SCENARIO TEST RESULTS")
print("=" * 55)

for scenario in scenarios:

    # Reset all roads to Open
    road_conditions.clear()
    road_conditions.update({
        road: "Open" for road in original_conditions
    })

    KNOWLEDGE_BASE["RoadStatus"].clear()

    for a, b, distance in ROADS:
        add_fact("RoadStatus", (a, b, "Open"))
        add_fact("RoadStatus", (b, a, "Open"))

    print("\n" + scenario["name"])
    print("-" * 55)

    if not scenario["reports"]:
        print("Road conditions: All roads are open.")

    for town_a, town_b, status in scenario["reports"]:

        affected = update_scenario_road(
            town_a, town_b, status
        )

        if len(affected) == 1:
            print(
                f"Reported: {affected[0]} is {status.lower()}."
            )
        else:
            print(
                f"Reported route: {town_a} to {town_b} "
                f"is {status.lower()}."
            )

            print(
                "Affected sections: " + ", ".join(affected)
            )

    route, distance = scenario_agent_decision(
        "Nairobi", "Kisumu"
    )

    if route:
        print("Agent decision: Alternative safe route found.")
        print("Recommended route:", " -> ".join(route))
        print("Total distance:", distance, "km")

    else:
        print("Agent decision: No safe route is available.")
        print("The trailer cannot proceed.")

    print("-" * 55)

road_conditions.clear()
road_conditions.update(original_conditions)

KNOWLEDGE_BASE["RoadStatus"].clear()
KNOWLEDGE_BASE["RoadStatus"].update(original_road_facts)

print("\nScenario testing completed.")


SELF-DRIVING TRAILER: SCENARIO TEST RESULTS

Scenario 1: All roads are open
-------------------------------------------------------
Road conditions: All roads are open.
Agent decision: Alternative safe route found.
Recommended route: Nairobi -> Limuru -> Naivasha -> Gilgil -> Nakuru -> Londiani -> Kericho -> Kisumu
Total distance: 355 km
-------------------------------------------------------

Scenario 2: Naivasha to Nakuru is closed
-------------------------------------------------------
Reported route: Naivasha to Nakuru is closed.
Affected sections: Naivasha to Gilgil, Gilgil to Nakuru
Agent decision: Alternative safe route found.
Recommended route: Nairobi -> Narok -> Bomet -> Kericho -> Kisumu
Total distance: 375 km
-------------------------------------------------------

Scenario 3: Narok to Kericho is closed
-------------------------------------------------------
Reported route: Narok to Kericho is closed.
Affected sections: Narok to Bomet, Bomet to Kericho
Agent decision: Alter

## Step 9: Debug the Knowledge Base

The purpose of debugging is to verify that the knowledge stored in the Knowledge Base (KB) is accurate, complete and consistent with the original road network.

Since the self-driving trailer agent uses this knowledge to make navigation decisions, incorrect or missing information could lead to incorrect route recommendations.

The following tests are performed:

**1. Town Records**

Checks whether all 15 towns in the original road network have been correctly added to the Knowledge Base. This ensures that the agent can recognise every town in its environment.

**2. Road Connections**

Checks whether all 16 physical road connections have been stored in both directions, giving 32 directed connections. This ensures that the agent can identify valid connections when travelling between towns.

**3. Road Distances**

Verifies that the distances stored in the Knowledge Base match the original road data. Accurate distances are important because the agent uses them to calculate the shortest route.

**4. Town Region Assignments**

Checks whether every town has been assigned to the correct geographical region. This ensures that the geographical information stored in the Knowledge Base is complete and accurate.

**5. Road Condition Records**

Checks whether the road conditions stored in the Knowledge Base match the current road conditions. This ensures that changes such as road closures and unsafe roads are correctly reflected in the agent's knowledge.

**6. Current Location**

Verifies that the agent's current location in the Knowledge Base matches its current state. This ensures that the agent knows where its journey begins.

**7. Destination**

Checks whether the destination stored in the Knowledge Base matches the agent's intended destination. This ensures that route planning is directed towards the correct location.

**8. Valid Road Conditions**

Checks that every road has one of the recognised conditions: Open, Closed or Unsafe. This prevents unsupported road conditions from affecting the agent's decisions.

**9. Positive Road Distances**

Checks that all road distances are greater than zero. This prevents invalid distance values from affecting route calculations.

Each test produces either a PASS or FAIL result. A PASS indicates that the information is correct, while a FAIL indicates that an inconsistency or invalid value has been detected.

The debugging process helps establish that the Knowledge Base contains reliable information for the agent's decision-making process.

In [9]:

def debug_knowledge_base():

    print("========== KNOWLEDGE BASE DEBUGGING ==========")

    checks = []

    # 1. Check town records
    expected_towns = set(graph.keys())
    actual_towns = KNOWLEDGE_BASE["Town"]

    checks.append((
        "Town records",
        actual_towns == expected_towns
    ))

    # 2. Check road connections in both directions
    expected_connections = set()

    for town_a, town_b, distance in ROADS:
        expected_connections.add((town_a, town_b))
        expected_connections.add((town_b, town_a))

    checks.append((
        "Road connections",
        KNOWLEDGE_BASE["Connected"] == expected_connections
    ))

    # 3. Check road distances
    expected_distances = set()

    for town_a, town_b, distance in ROADS:
        expected_distances.add((town_a, town_b, distance))
        expected_distances.add((town_b, town_a, distance))

    checks.append((
        "Road distances",
        KNOWLEDGE_BASE["Distance"] == expected_distances
    ))

    # 4. Check geographical region assignments
    expected_regions = {
        (town, region)
        for town, region in REGIONS.items()
    }

    checks.append((
        "Town region assignments",
        KNOWLEDGE_BASE["LocatedIn"] == expected_regions
    ))

    # 5. Check road condition records
    expected_statuses = set()

    for town_a, town_b, distance in ROADS:
        status = road_conditions[frozenset([town_a, town_b])]

        expected_statuses.add((town_a, town_b, status))
        expected_statuses.add((town_b, town_a, status))

    checks.append((
        "Road condition records",
        KNOWLEDGE_BASE["RoadStatus"] == expected_statuses
    ))

    # 6. Check current location
    checks.append((
        "Current location",
        KNOWLEDGE_BASE["At"] == {
            AGENT_STATE["current_location"]
        }
    ))

    # 7. Check destination
    checks.append((
        "Destination",
        KNOWLEDGE_BASE["Destination"] == {
            AGENT_STATE["destination"]
        }
    ))

    # 8. Check valid road conditions
    valid_conditions = {"Open", "Closed", "Unsafe"}

    checks.append((
        "Valid road conditions",
        all(
            status in valid_conditions
            for status in road_conditions.values()
        )
    ))

    # 9. Check positive road distances
    checks.append((
        "Positive road distances",
        all(
            distance > 0
            for _, _, distance in ROADS
        )
    ))

    # Display debugging results
    passed = 0

    for check_name, result in checks:

        if result:
            print(f"[PASS] {check_name}")
            passed += 1
        else:
            print(f"[FAIL] {check_name}")

    print("----------------------------------------------")
    print(f"Checks passed: {passed}/{len(checks)}")

    if passed == len(checks):
        print("Knowledge Base is consistent.")
    else:
        print("Knowledge Base contains inconsistencies.")

    print("==============================================")


# Run the debugging process
debug_knowledge_base()


========== KNOWLEDGE BASE DEBUGGING ==========
[PASS] Town records
[PASS] Road connections
[PASS] Road distances
[PASS] Town region assignments
[PASS] Road condition records
[PASS] Current location
[PASS] Destination
[PASS] Valid road conditions
[PASS] Positive road distances
----------------------------------------------
Checks passed: 9/9
Knowledge Base is consistent.


## Step 10: Final Reflection

### Overview

This practical exercise involved developing a Knowledge-Based Agent for a self-driving trailer operating within a simulated Kenyan road network. The agent was designed to navigate from a starting location to a destination while considering road distances, connectivity and changing road conditions.

The Knowledge Engineering process was followed to identify the problem, assemble relevant knowledge, define the vocabulary, encode general rules, populate the Knowledge Base, process queries, make decisions and debug the system.

### How the Knowledge-Based Agent Works

The agent uses a Knowledge Base containing information about towns, road connections, distances, geographical regions and road conditions.

It applies predefined rules to determine which roads are safe to use. Using Uniform Cost Search (UCS), it identifies the shortest available route to the destination while avoiding closed and unsafe roads.

The agent can also process user-provided road-condition reports, answer questions about the road network and adjust its route recommendations when conditions change.

### Challenges Encountered

Several challenges were encountered during the development process:

1. **Representing the road network:** Ensuring that all towns and road connections were correctly represented in the Knowledge Base, including connections in both directions.

2. **Route selection:** Ensuring that the agent could identify alternative routes when its preferred route became unavailable.

3. **Processing user queries:** Recognising town names and interpreting questions involving direct and indirect road connections.

4. **Debugging the Knowledge Base:** Verifying that stored facts, road distances, road conditions and agent information remained consistent throughout the execution of the system.

### Possible Improvements

1. **Real-time traffic information:** Integrate live traffic updates to help the agent avoid congestion and delays.

2. **GPS integration:** Use actual GPS information to determine the trailer's current location.

3. **Dynamic road updates:** Automatically obtain road closure and safety information from reliable external sources.

4. **More advanced decision-making:** Incorporate additional factors such as fuel consumption, travel time, road quality and weather conditions when selecting routes.

5. **Improved user interface:** Develop a graphical interface with a map to display the road network, current location and recommended routes.

### Conclusion

This exercise demonstrated how Knowledge Engineering can be used to develop an intelligent agent capable of representing knowledge, reasoning about its environment and making decisions based on available information.

The developed agent successfully demonstrates route planning, knowledge-based decision-making, query processing and adaptation to changing road conditions within the simulated road network.